In [1]:
from dataloader import build_loaders
from model import build_model

train_loader, val_loader = build_loaders(
    data_root   = '/fs/vulcan-projects/fsh_track/jason-bhargav-temp/CMSC472-Final/data',
    val_subject = 1,
    dataset     = 'SEED-IV',
    n_per_class = 8,
)

# IMPORTANT: Samples=5 (the 5 frequency bands), not 200
model = build_model(nb_classes=4, Chans=62, Samples=5)



Building loaders  [cross_subject]  dataset=SEED-IV  feature=de_LDS
  Loading subject 01/15 … 2505 windows
  Loading subject 02/15 … 2505 windows
  Loading subject 03/15 … 2505 windows
  Loading subject 04/15 … 2505 windows
  Loading subject 05/15 … 2505 windows
  Loading subject 06/15 … 2505 windows
  Loading subject 07/15 … 2505 windows
  Loading subject 08/15 … 2505 windows
  Loading subject 09/15 … 2505 windows
  Loading subject 10/15 … 2505 windows
  Loading subject 11/15 … 2505 windows
  Loading subject 12/15 … 2505 windows
  Loading subject 13/15 … 2505 windows
  Loading subject 14/15 … 2505 windows
  Loading subject 15/15 … 2505 windows

  Train : (35070, 62, 5)  labels [9492 9562 8610 7406]
  Val   : (2505, 62, 5)  labels [678 683 615 529]

  Batch size    : 32  (8/class × 4 classes)
  Train batches : 925
  Val   batches : 79



In [2]:
train_loader, val_loader = build_loaders(
    data_root   = '/fs/vulcan-projects/fsh_track/jason-bhargav-temp/CMSC472-Final/data',
    val_subject = 1,
    dataset     = 'SEED-IV',
    window_sec  = 1.0,
    sfreq       = 200,
    n_per_class = 8,
)


Building loaders  [cross_subject]  dataset=SEED-IV  feature=de_LDS
  Loading subject 01/15 … 2505 windows
  Loading subject 02/15 … 2505 windows
  Loading subject 03/15 … 2505 windows
  Loading subject 04/15 … 2505 windows
  Loading subject 05/15 … 2505 windows
  Loading subject 06/15 … 2505 windows
  Loading subject 07/15 … 2505 windows
  Loading subject 08/15 … 2505 windows
  Loading subject 09/15 … 2505 windows
  Loading subject 10/15 … 2505 windows
  Loading subject 11/15 … 2505 windows
  Loading subject 12/15 … 2505 windows
  Loading subject 13/15 … 2505 windows
  Loading subject 14/15 … 2505 windows
  Loading subject 15/15 … 2505 windows

  Train : (35070, 62, 5)  labels [9492 9562 8610 7406]
  Val   : (2505, 62, 5)  labels [678 683 615 529]

  Batch size    : 32  (8/class × 4 classes)
  Train batches : 925
  Val   batches : 79



In [ ]:
import torch
from model import build_model, EEGDataset, BalancedBatchSampler, Trainer
from losses import ClassificationLoss, ContrastiveLoss

model     = build_model(nb_classes=4, Chans=62, Samples=5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

trainer = Trainer(
    model, ClassificationLoss(), ContrastiveLoss(temperature=0.1),
    optimizer, lambda_con=0.5, warmup_epochs=5, device='cuda'
)
history = trainer.fit(train_loader, val_loader, epochs=50)

Epoch 001/50  train_loss=3.4527  cls=0.0000  con=3.4527  acc=0.246  |  val_loss=3.4247  val_acc=0.240
Epoch 002/50  train_loss=3.4347  cls=0.0000  con=3.4347  acc=0.244  |  val_loss=3.4255  val_acc=0.193
Epoch 003/50  train_loss=3.4328  cls=0.0000  con=3.4328  acc=0.241  |  val_loss=3.4274  val_acc=0.262
Epoch 004/50  train_loss=3.4267  cls=0.0000  con=3.4267  acc=0.249  |  val_loss=3.4273  val_acc=0.273
Epoch 005/50  train_loss=2.9931  cls=1.2846  con=3.4170  acc=0.379  |  val_loss=3.1055  val_acc=0.267
Epoch 006/50  train_loss=2.9408  cls=1.2368  con=3.4081  acc=0.418  |  val_loss=3.0695  val_acc=0.438
Epoch 007/50  train_loss=2.9067  cls=1.2065  con=3.4004  acc=0.445  |  val_loss=3.0842  val_acc=0.378
Epoch 008/50  train_loss=2.8722  cls=1.1772  con=3.3899  acc=0.464  |  val_loss=3.0816  val_acc=0.339
Epoch 009/50  train_loss=2.8418  cls=1.1525  con=3.3786  acc=0.477  |  val_loss=3.0434  val_acc=0.374
Epoch 010/50  train_loss=2.8340  cls=1.1450  con=3.3779  acc=0.487  |  val_loss=3.

In [ ]:
import os

# Save
save_path = 'checkpoints/ContrastiveLoss_0.1.pt'
os.makedirs('checkpoints', exist_ok=True)

torch.save({
    'model_state_dict'    : model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history'             : history,
}, save_path)

print(f"Model saved to {save_path}")


In [ ]:
checkpoint = torch.load('checkpoints/eegnet_seediv.pt', map_location='cuda')

model = build_model(nb_classes=4, Chans=62, Samples=5)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()